# Notebook 04 — Context Assembly & gemma3 Generation
**Layer:** Generation  
**Model:** `gemma3` via local ollama — no API key, no rate limits, no internet  
**Inputs:** Cypher result (NB03) · FAISS index (NB02) · `feature_metadata_20260403.json`  
**Outputs:** Structured JSON answer with price estimate, comparables, factors, caveats

## Setup — run once
```bash
ollama pull gemma3
```

## 4.1 Imports, ollama client, load indexes and pipeline

In [18]:
import json
import re
import numpy as np
import faiss
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import normalize
from neo4j import GraphDatabase
import ollama

# Local ollama — no API key, no rate limits, no internet required
OLLAMA_MODEL_Chat = "gemma3"
OLLAMA_MODEL = "nomic-embed-text"

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
CSV_PATH    = FEATURE_DIR / "hdb_feature_table_20260412.csv"

INDEX_DIR   = Path("03_vector_index")
META_PATH   = FEATURE_DIR / "feature_metadata_20260412.json"

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"   # <-- update

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# Load EMBED_DIM from NB02 config
with open(INDEX_DIR / "embed_config.json") as f:
    cfg = json.load(f)
EMBED_DIM = cfg["embed_dim"]
print(f"Embedding method : {cfg['method']}")
print(f"Model chat" + OLLAMA_MODEL_Chat)
print(f"EMBED_DIM        : {EMBED_DIM}")

# Load FAISS indexes and metadata
index_pre  = faiss.read_index(str(INDEX_DIR / "index_pre2023.faiss"))
index_post = faiss.read_index(str(INDEX_DIR / "index_post2023.faiss"))
meta_pre   = pd.read_parquet(INDEX_DIR / "meta_pre2023.parquet")
meta_post  = pd.read_parquet(INDEX_DIR / "meta_post2023.parquet")

with open(META_PATH) as f:
    feature_meta = json.load(f)

# Price drift constants from README section 3
TRAIN_MEAN_PRICE = 468788
TEST_MEAN_PRICE  = 614711
TEMPORAL_SPLIT   = 2023

print(f"pre-2023  index  : {index_pre.ntotal:,} vectors")
print(f"post-2023 index  : {index_post.ntotal:,} vectors")
print(f"ollama model     : {OLLAMA_MODEL}")

Embedding method : ollama_gemma3
Model chatgemma3
EMBED_DIM        : 768
pre-2023  index  : 178,589 vectors
post-2023 index  : 82,110 vectors
ollama model     : nomic-embed-text


## 4.2 Ollama wrapper and routing helpers

In [19]:
def call_ollama(prompt: str, system: str = "", fmt: str = "") -> str:
    """
    Call gemma3 via local ollama. No API key, no rate limits, no internet.
    - prompt : user message
    - system : optional system instruction
    - fmt    : pass "json" to request JSON-only output
    Returns the response text string.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": OLLAMA_MODEL_Chat, "messages": messages}
    if fmt == "json":
        kwargs["format"] = "json"

    response = ollama.chat(**kwargs)
    return response.message.content.strip()


VALID_TOWNS = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT BATOK","BUKIT MERAH","BUKIT PANJANG",
    "BUKIT TIMAH","CENTRAL AREA","CHOA CHU KANG","CLEMENTI","GEYLANG","HOUGANG",
    "JURONG EAST","JURONG WEST","KALLANG/WHAMPOA","MARINE PARADE","PASIR RIS",
    "PUNGGOL","QUEENSTOWN","SEMBAWANG","SENGKANG","SERANGOON","TAMPINES",
    "TOA PAYOH","WOODLANDS","YISHUN",
]
VALID_FLAT_TYPES = [
    "1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"
]

CLASSIFIER_SYSTEM = """You are a query classifier for an HDB flat database system.
Classify into ONE of: PRICE_ESTIMATION, NEIGHBOURHOOD, SCHOOL_CATCHMENT,
INVESTMENT_TEMPORAL, LEASE_ADVISORY.
Extract slots: town, flat_type, room_count, budget_sgd_min, budget_sgd_max,
year_min, year_max, lease_years_max (null if not mentioned).
Respond ONLY with valid JSON:
{"intent": "...", "slots": {"town": null, "flat_type": null, "room_count": null,
"budget_sgd_min": null, "budget_sgd_max": null, "year_min": null,
"year_max": null, "lease_years_max": null}}
"""

CYPHER_TEMPLATES = {
    "PRICE_ESTIMATION": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town) AND ($flat_type IS NULL OR f.flat_type=$flat_type)
  AND ($room_count IS NULL OR f.room_count=$room_count)
  AND ($budget_sgd_min IS NULL OR f.resale_price>=$budget_sgd_min)
  AND ($budget_sgd_max IS NULL OR f.resale_price<=$budget_sgd_max)
RETURN percentileCont(f.resale_price,0.5) AS median_price,
       avg(f.resale_price) AS avg_price, stDev(f.resale_price) AS std_price,
       count(f) AS tx_count, min(f.transaction_year) AS year_min,
       max(f.transaction_year) AS year_max""",
    "NEIGHBOURHOOD": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town) WHERE ($town IS NULL OR t.name=$town)
RETURN t.name AS town, avg(f.dist_to_mrt_m) AS avg_mrt_dist_m,
       avg(f.dist_to_foodcourt_m) AS avg_foodcourt_dist_m,
       avg(f.mall_count_3km) AS avg_mall_count_3km, count(f) AS flat_count""",
    "SCHOOL_CATCHMENT": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town)
  AND ($budget_sgd_max IS NULL OR f.resale_price<=$budget_sgd_max)
RETURN t.name AS town,
       avg(f.primary_school_quality_1km_weighted) AS avg_school_quality,
       avg(f.school_count_1km) AS avg_schools_in_1km,
       avg(f.resale_price) AS avg_price, count(f) AS flat_count
ORDER BY avg_school_quality DESC""",
    "INVESTMENT_TEMPORAL": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town)
  AND ($year_min IS NULL OR f.transaction_year>=$year_min)
  AND ($year_max IS NULL OR f.transaction_year<=$year_max)
RETURN t.name AS town, f.transaction_year AS year,
       avg(f.resale_price) AS avg_price, count(f) AS tx_count
ORDER BY t.name, f.transaction_year""",
    "LEASE_ADVISORY": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town)
  AND ($lease_years_max IS NULL OR f.lease_remaining_years<=$lease_years_max)
RETURN CASE WHEN f.lease_remaining_years<50 THEN '<50 years'
            WHEN f.lease_remaining_years<70 THEN '50-69 years'
            ELSE '70+ years' END AS lease_band,
       avg(f.resale_price) AS avg_price, count(f) AS tx_count
ORDER BY lease_band"""
}

def classify_query(user_query: str) -> dict:
    prompt = "Classify this query and extract slots:\n\n" + user_query
    raw    = call_ollama(prompt, system=CLASSIFIER_SYSTEM, fmt="json")
    return json.loads(re.sub(r"```json|```", "", raw).strip())

def slots_to_params(slots: dict) -> dict:
    town      = slots.get("town")
    flat_type = slots.get("flat_type")
    if town and town.upper() not in VALID_TOWNS:               town = None
    if flat_type and flat_type.upper() not in VALID_FLAT_TYPES: flat_type = None
    return {
        "town": town.upper() if town else None,
        "flat_type": flat_type.upper() if flat_type else None,
        "room_count": slots.get("room_count"),
        "budget_sgd_min": slots.get("budget_sgd_min"),
        "budget_sgd_max": slots.get("budget_sgd_max"),
        "year_min": slots.get("year_min"),
        "year_max": slots.get("year_max"),
        "lease_years_max": slots.get("lease_years_max"),
    }

print("Routing helpers defined.")

Routing helpers defined.


## 4.3 Vector retrieval using ollama embeddings

In [20]:
def embed_query(query_text: str) -> np.ndarray:
    # Embed a single query string using ollama gemma3
    resp = ollama.embed(model=OLLAMA_MODEL, input=[query_text])
    vec  = np.array(resp.embeddings[0], dtype=np.float32).reshape(1, -1)
    return normalize(vec, norm="l2").astype(np.float32)   # (1, EMBED_DIM)


def is_recent_query(slots: dict) -> bool:
    year_min = slots.get("year_min")
    year_max = slots.get("year_max")
    if year_min and year_min >= TEMPORAL_SPLIT: return True
    if year_max and year_max >= TEMPORAL_SPLIT: return True
    return False


def vector_retrieve(query_text: str, is_recent: bool, top_k: int = 20) -> pd.DataFrame:
    q_vec = embed_query(query_text)
    faiss.normalize_L2(q_vec)
    if is_recent:
        D, I = index_post.search(q_vec, top_k);  meta = meta_post
    else:
        D, I = index_pre.search(q_vec, top_k);   meta = meta_pre
    hits = meta.iloc[I[0]].copy()
    hits["similarity_score"] = D[0]
    return hits.reset_index(drop=True)


print("embed_query() and vector_retrieve() defined.")

embed_query() and vector_retrieve() defined.


## 4.4 Context assembly and gemma3 generation

In [21]:
FACTOR_MAP = {
    "PRICE_ESTIMATION":   ["level_mid","lease_remaining_years","floor_area_sqm","room_count","dist_to_mrt_m"],
    "NEIGHBOURHOOD":      ["dist_to_mrt_m","dist_to_highway_m","dist_to_foodcourt_m","mall_count_3km"],
    "SCHOOL_CATCHMENT":   ["dist_to_nearest_school_m","school_count_1km","primary_school_quality_1km_weighted"],
    "INVESTMENT_TEMPORAL":["transaction_year","resale_price"],
    "LEASE_ADVISORY":     ["lease_remaining_years","resale_price"],
}

def assemble_context(user_query, intent, cypher_records, vector_hits, is_recent):
    ctx = ["=== GRAPH AGGREGATES (from Neo4j) ===",
           json.dumps(cypher_records[:10], indent=2, default=str),
           "\n=== COMPARABLE TRANSACTIONS (top-5) ==="]
    COMP = ["address_key","town","flat_type","floor_area_sqm",
            "level_mid","lease_remaining_years","resale_price","transaction_year"]
    ctx.append(vector_hits.head(5)[COMP].to_string(index=False))
    defs = [f"  {f}: {feature_meta.get('features',{}).get(f,{}).get('description','')}"
            for f in FACTOR_MAP.get(intent,[]) if f in feature_meta.get("features",{})]
    if defs:
        ctx.append("\n=== FEATURE DEFINITIONS ==="); ctx.extend(defs)
    if is_recent:
        ctx.append(
            f"\n=== TEMPORAL CONTEXT ===\n"
            f"Post-2023 data. Mean price ${TEST_MEAN_PRICE:,} vs "
            f"pre-2023 ${TRAIN_MEAN_PRICE:,} (+31%). Include caveat."
        )
    return "\n".join(ctx)


GENERATION_SYSTEM = """You are an expert HDB property advisor for Singapore.
RULES:
1. Answer ONLY using GRAPH AGGREGATES and COMPARABLE TRANSACTIONS provided.
2. Do NOT introduce any price figures, names or locations not in the context.
3. Cite the transaction_year range of comparables used.
4. Output ONLY valid JSON, no markdown, no explanation.
SCHEMA:
{"estimate_sgd": integer_or_null, "low_sgd": integer_or_null,
 "high_sgd": integer_or_null, "basis_count": integer,
 "key_factors": [{"factor_name": str, "value": str, "impact": str}],
 "caveats": [str], "comparable_years_range": str, "narrative": str}
"""

def generate_answer(user_query: str, context: str) -> dict:
    prompt = context + "\n\nUSER QUERY: " + user_query
    raw    = call_ollama(prompt, system=GENERATION_SYSTEM, fmt="json")
    return json.loads(re.sub(r"```json|```", "", raw).strip())

print("assemble_context() and generate_answer() defined.")

assemble_context() and generate_answer() defined.


## 4.5 Full pipeline and batch test

In [22]:
#OLLAMA_MODEL        = "nomic-embed-text"
def run_pipeline(user_query: str) -> dict:
    classified    = classify_query(user_query)
    intent        = classified["intent"]
    slots         = classified["slots"]
    params        = slots_to_params(slots)
    recent        = is_recent_query(slots)

    with driver.session() as session:
        cypher_records = [dict(r) for r in
                          session.run(CYPHER_TEMPLATES[intent], **params)]

    vector_hits = vector_retrieve(user_query, is_recent=recent, top_k=20)
    context     = assemble_context(user_query, intent, cypher_records,
                                   vector_hits, recent)
    answer      = generate_answer(user_query, context)

    return {"query": user_query, "intent": intent, "slots": slots,
            "is_recent": recent, "cypher_rows": len(cypher_records),
            "vector_hits": len(vector_hits), "answer": answer}


test_queries = [
    "How much is a 4-room flat in Bishan worth?",
    "What amenities are near Ang Mo Kio flats?",
    "Which areas have the best primary schools under $700K?",
    "Which town had the best price growth from 2020 to 2025?",
    "Should I buy a flat with only 55 years of lease left?",
]
for q in test_queries:
    r = run_pipeline(q)
    print(f"Intent: {r['intent']}")
    ans = r["answer"]
    if ans.get("estimate_sgd"):
        print(f"  Estimate : ${ans['estimate_sgd']:,}")
    else:
        print(f"  Narrative: {ans.get('narrative','')[:80]}")
    print(f"  Caveats  : {ans.get('caveats',[])}")
    print()

driver.close()
print("Notebook 04 complete.")

Intent: PRICE_ESTIMATION
  Narrative: Due to the lack of data for Bishan 4-room flats within the provided transaction 
  Caveats  : ['No transactions for Bishan were found within the provided dataset.', 'The dataset is limited to only 5 comparable transactions.']

Intent: NEIGHBOURHOOD
  Narrative: 
  Caveats  : ['No amenities were identified in the provided transaction data.']

Intent: SCHOOL_CATCHMENT
  Narrative: 
  Caveats  : []

Intent: PRICE_ESTIMATION
  Narrative: 
  Caveats  : []

Intent: LEASE_ADVISORY
  Narrative: Based on the available data, properties with 55 years or less of lease remaining
  Caveats  : ['Limited comparable data with short lease remaining years.']

Notebook 04 complete.
